# 06 — Semantic Vector Search

This notebook implements the first retrieval stage of **CLARA**.

The previous notebooks transformed the legal corpus from PDF documents into structured articles, retrieval chunks, semantic embeddings, and finally a persistent Qdrant vector collection. This notebook verifies that the indexed corpus can be searched effectively using natural-language questions.

A user question is converted into an embedding using the same **BGE-M3** model used during indexing. The resulting query vector is then compared with the vectors stored in Qdrant, returning the most semantically relevant legal chunks.

The retrieval flow is:

```text
Natural-Language Question
          │
          ▼
       BGE-M3
          │
          ▼
   Query Embedding
          │
          ▼
       Qdrant
          │
          ▼
  Top-K Legal Chunks
          │
          ▼
 Metadata + Source Text
```

## 1. Imports

This block loads the libraries required for HTTP communication with Ollama and Qdrant, data handling, timing, and result inspection.

In [1]:
import json

import pandas as pd
import requests


## 2. Retrieval Configuration

The main retrieval parameters are defined here, including the Ollama endpoint, Qdrant endpoint, embedding model, vector collection, and number of results to retrieve.

The embedding model must be identical to the model used in `04_embedding_generation.ipynb`. Query and document vectors can only be compared meaningfully when they belong to the same embedding space.

In [2]:
OLLAMA_URL = "http://localhost:11434"
EMBEDDING_MODEL = "bge-m3"

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "clara_bge_m3"

TOP_K = 5

print(f"Ollama:     {OLLAMA_URL}")
print(f"Modelo:     {EMBEDDING_MODEL}")
print(f"Qdrant:     {QDRANT_URL}")
print(f"Collection: {COLLECTION_NAME}")
print(f"Top-K:      {TOP_K}")


Ollama:     http://localhost:11434
Modelo:     bge-m3
Qdrant:     http://localhost:6333
Collection: clara_bge_m3
Top-K:      5


## Verificar serviços


## 2. Retrieval Configuration

The main retrieval parameters are defined here, including the Ollama endpoint, Qdrant endpoint, embedding model, vector collection, and number of results to retrieve.

The embedding model must be identical to the model used in `04_embedding_generation.ipynb`. Query and document vectors can only be compared meaningfully when they belong to the same embedding space.

In [3]:
def check_service(
    url: str,
    timeout: int = 10
) -> bool:
    try:
        response = requests.get(
            url,
            timeout=timeout
        )
        response.raise_for_status()
        return True

    except Exception:
        return False


ollama_ok = check_service(
    f"{OLLAMA_URL}/api/tags"
)

qdrant_ok = check_service(
    f"{QDRANT_URL}/collections"
)

print(
    "Ollama:",
    "OK" if ollama_ok else "INDISPONÍVEL"
)

print(
    "Qdrant:",
    "OK" if qdrant_ok else "INDISPONÍVEL"
)


Ollama: OK
Qdrant: OK


## Gerar embedding da pergunta


## 4. Query Embedding Generation

This function converts a natural-language question into a dense semantic vector using **BGE-M3** through Ollama.

Using the same embedding model for documents and questions allows semantic similarity to be calculated even when the wording of the question differs from the wording used in the legislation.

In [4]:
def generate_embedding(
    text: str,
    model: str = EMBEDDING_MODEL
) -> list[float]:

    if not text or not text.strip():
        raise ValueError(
            "A pergunta não pode estar vazia."
        )

    response = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={
            "model": model,
            "input": text
        },
        timeout=120
    )

    response.raise_for_status()

    data = response.json()

    embeddings = data.get("embeddings")

    if not embeddings:
        raise ValueError(
            "Ollama não devolveu embeddings."
        )

    return embeddings[0]


### 4.1 Testing Query Embedding

A representative legal question is embedded to validate the query-side embedding process.

The resulting vector can also be inspected to confirm that its dimensionality matches the vectors stored in the Qdrant collection.

In [5]:
test_question = (
    "Como são tributados os rendimentos prediais?"
)

test_vector = generate_embedding(
    test_question
)

print(f"Pergunta: {test_question}")
print(f"Dimensão: {len(test_vector)}")


Pergunta: Como são tributados os rendimentos prediais?
Dimensão: 1024


## Pesquisa vetorial

A função abaixo permite pesquisar globalmente ou restringir a pesquisa a um diploma específico.


## 4. Query Embedding Generation

This function converts a natural-language question into a dense semantic vector using **BGE-M3** through Ollama.

Using the same embedding model for documents and questions allows semantic similarity to be calculated even when the wording of the question differs from the wording used in the legislation.

In [6]:
def vector_search(
    question: str,
    top_k: int = 5,
    document_id: str | None = None
) -> list[dict]:

    query_vector = generate_embedding(
        question
    )

    search_payload = {
        "vector": query_vector,
        "limit": top_k,
        "with_payload": True,
        "with_vector": False
    }

    if document_id is not None:

        search_payload["filter"] = {
            "must": [
                {
                    "key": "document_id",
                    "match": {
                        "value": document_id
                    }
                }
            ]
        }

    response = requests.post(
        (
            f"{QDRANT_URL}/collections/"
            f"{COLLECTION_NAME}/points/search"
        ),
        json=search_payload,
        timeout=120
    )

    response.raise_for_status()

    return response.json()["result"]


## Pesquisa global


### 5.1 Top-K Retrieval

The number of returned results is controlled through the Top-K parameter.

Retrieving multiple candidates is important in a RAG system because the information required to answer a question may be distributed across several articles or chunks rather than contained in the single highest-ranked result.

In [7]:
question = (
    "Como são tributados os rendimentos prediais?"
)

results = vector_search(
    question,
    top_k=TOP_K
)

print(f"Pergunta: {question}\n")

for rank, result in enumerate(
    results,
    start=1
):

    payload = result["payload"]

    print("=" * 80)

    print(
        f"{rank}. "
        f"{payload['document_id']} — "
        f"{payload['article']}"
    )

    print(
        f"Título: {payload.get('article_title')}"
    )

    print(
        f"Páginas: "
        f"{payload['page_start']}"
        f"-{payload['page_end']}"
    )

    print(
        f"Score: {result['score']:.4f}"
    )

    print()

    print(
        payload["text"][:1200]
    )

    print()


Pergunta: Como são tributados os rendimentos prediais?

1. CIRS — Artigo 8.º
Título: Rendimentos da categoria F
Páginas: 32-33
Score: 0.7041

1 - Consideram-se rendimentos prediais as rendas dos prédios rústicos, urbanos e mistos pagas ou colocadas à disposição dos respetivos titulares, quando estes não optarem pela sua tributação no âmbito da categoria B. 2 - São havidas como rendas: 32 a) As importâncias relativas à cedência do uso do prédio ou de parte dele e aos serviços relacionados com aquela cedência; b) As importâncias relativas ao aluguer de maquinismos e mobiliários instalados no imóvel locado; c) A diferença, auferida pelo sublocador, entre a renda recebida do subarrendatário e a paga ao senhorio; d) As importâncias relativas à cedência do uso, total ou parcial, de bens imóveis, para quaisquer fins especiais, designadamente publicidade; e) As importâncias relativas à cedência do uso de partes comuns de prédios em regime de propriedade horizontal; f) As importâncias relativas

## Resultados em DataFrame


## 6. Inspecting Search Results

The retrieved results are inspected together with their similarity scores and legal metadata.

For each result, the most relevant information includes:

- similarity score;
- source document;
- article;
- article title;
- page range;
- chunk identifier;
- retrieved legal text.

This metadata provides traceability between the semantic search result and the original legislation.

In [8]:
def results_to_dataframe(
    results: list[dict]
) -> pd.DataFrame:

    rows = []

    for rank, result in enumerate(
        results,
        start=1
    ):

        payload = result["payload"]

        rows.append({
            "rank": rank,
            "score": result["score"],
            "document_id": payload[
                "document_id"
            ],
            "article": payload[
                "article"
            ],
            "article_title": payload.get(
                "article_title"
            ),
            "page_start": payload[
                "page_start"
            ],
            "page_end": payload[
                "page_end"
            ],
            "chunk_id": payload[
                "chunk_id"
            ],
            "text": payload[
                "text"
            ]
        })

    return pd.DataFrame(rows)


df_results = results_to_dataframe(
    results
)

df_results[
    [
        "rank",
        "score",
        "document_id",
        "article",
        "article_title",
        "page_start",
        "page_end",
        "chunk_id"
    ]
]


,rank,score,document_id,article,article_title,page_start,page_end,chunk_id
0,1,0.704147,CIRS,Artigo 8.º,Rendimentos da categoria F,32,33,CIRS_ART_0027_CHUNK_001
1,2,0.656335,EBF,Artigo 45.º-C,"(Aditado pelo Decreto-Lei n.º 97/2026, de 20 d...",79,81,EBF_ART_0128_CHUNK_001
2,3,0.651471,CIRS,Artigo 16.º,As modificações do Código,11,23,CIRS_ART_0017_CHUNK_013
3,4,0.638217,EBF,Artigo 74.º-A,Transferência de imóveis de alojamento local p...,106,107,EBF_ART_0183_CHUNK_001
4,5,0.631915,CIRS,Artigo 72.º,Taxas especiais,86,91,CIRS_ART_0108_CHUNK_002


## Pesquisa filtrada por diploma

Exemplo: pesquisar exclusivamente no CIRS.


### 5.1 Top-K Retrieval

The number of returned results is controlled through the Top-K parameter.

Retrieving multiple candidates is important in a RAG system because the information required to answer a question may be distributed across several articles or chunks rather than contained in the single highest-ranked result.

In [9]:
question = (
    "Como são tributados os rendimentos prediais?"
)

document_filter = "CIRS"

filtered_results = vector_search(
    question,
    top_k=TOP_K,
    document_id=document_filter
)

df_filtered = results_to_dataframe(
    filtered_results
)

df_filtered[
    [
        "rank",
        "score",
        "document_id",
        "article",
        "article_title",
        "page_start",
        "page_end"
    ]
]


,rank,score,document_id,article,article_title,page_start,page_end
0,1,0.704147,CIRS,Artigo 8.º,Rendimentos da categoria F,32,33
1,2,0.651471,CIRS,Artigo 16.º,As modificações do Código,11,23
2,3,0.631915,CIRS,Artigo 72.º,Taxas especiais,86,91
3,4,0.629810,CIRS,Artigo 72.º,Taxas especiais,86,91
4,5,0.598820,CIRS,Artigo 41.º,Deduções,68,69


## Construção do contexto para o LLM

O contexto combina os chunks recuperados com as respetivas referências jurídicas.


## 9. Building Retrieval Context

The selected legal chunks are combined into a textual context suitable for use by a Large Language Model.

This step establishes the bridge between retrieval and generation: Qdrant identifies the relevant legislation, and the retrieved passages become the evidence supplied to the language model.

In [10]:
def build_context(
    results: list[dict]
) -> str:

    context_blocks = []

    for rank, result in enumerate(
        results,
        start=1
    ):

        payload = result["payload"]

        reference = (
            f"[Fonte {rank}] "
            f"{payload['document_id']} — "
            f"{payload['article']}"
        )

        if payload.get("article_title"):
            reference += (
                f" — {payload['article_title']}"
            )

        reference += (
            f" — páginas "
            f"{payload['page_start']}"
            f"-{payload['page_end']}"
        )

        block = (
            f"{reference}\n"
            f"{payload['text']}"
        )

        context_blocks.append(block)

    return "\n\n".join(
        context_blocks
    )


## 9. Building Retrieval Context

The selected legal chunks are combined into a textual context suitable for use by a Large Language Model.

This step establishes the bridge between retrieval and generation: Qdrant identifies the relevant legislation, and the retrieved passages become the evidence supplied to the language model.

In [11]:
context = build_context(
    results
)

print(context[:6000])


[Fonte 1] CIRS — Artigo 8.º — Rendimentos da categoria F — páginas 32-33
1 - Consideram-se rendimentos prediais as rendas dos prédios rústicos, urbanos e mistos pagas ou colocadas à disposição dos respetivos titulares, quando estes não optarem pela sua tributação no âmbito da categoria B. 2 - São havidas como rendas: 32 a) As importâncias relativas à cedência do uso do prédio ou de parte dele e aos serviços relacionados com aquela cedência; b) As importâncias relativas ao aluguer de maquinismos e mobiliários instalados no imóvel locado; c) A diferença, auferida pelo sublocador, entre a renda recebida do subarrendatário e a paga ao senhorio; d) As importâncias relativas à cedência do uso, total ou parcial, de bens imóveis, para quaisquer fins especiais, designadamente publicidade; e) As importâncias relativas à cedência do uso de partes comuns de prédios em regime de propriedade horizontal; f) As importâncias relativas à constituição, a título oneroso, de direitos reais de gozo temporár

## Função integrada de recuperação

Esta função reúne pesquisa, tabela de resultados e contexto numa única chamada.


## 5. Semantic Search Function

This block defines the semantic retrieval operation.

The query embedding is sent to Qdrant, which compares it with the indexed legal vectors and returns the highest-ranked chunks according to vector similarity.

The retrieval result includes the similarity score and the payload metadata required to identify the original legal source.

In [12]:
def retrieve(
    question: str,
    top_k: int = 5,
    document_id: str | None = None
) -> dict:

    results = vector_search(
        question=question,
        top_k=top_k,
        document_id=document_id
    )

    return {
        "question": question,
        "document_filter": document_id,
        "top_k": top_k,
        "results": results,
        "dataframe": results_to_dataframe(
            results
        ),
        "context": build_context(
            results
        )
    }


### 5.1 Top-K Retrieval

The number of returned results is controlled through the Top-K parameter.

Retrieving multiple candidates is important in a RAG system because the information required to answer a question may be distributed across several articles or chunks rather than contained in the single highest-ranked result.

In [13]:
retrieval = retrieve(
    question=(
        "Que benefícios fiscais existem "
        "para imóveis destinados a habitação?"
    ),
    top_k=5
)

retrieval["dataframe"][
    [
        "rank",
        "score",
        "document_id",
        "article",
        "article_title"
    ]
]


,rank,score,document_id,article,article_title
0,1,0.657290,EBF,Artigo 44.º-B,Outros benefícios com caráter ambiental atribu...
1,2,0.643414,EBF,Artigo 71.º-A,Incentivos ao arrendamento habitacional a cust...
2,3,0.639570,EBF,Artigo 45.º-A,Prédios urbanos destinados ao Programa de Apoi...
3,4,0.635775,CIMI,Artigo 11.º-B,Isenção aplicável a terrenos para construção d...
4,5,0.626441,EBF,Artigo 44.º-B,Outros benefícios com caráter ambiental atribu...


## 9. Building Retrieval Context

The selected legal chunks are combined into a textual context suitable for use by a Large Language Model.

This step establishes the bridge between retrieval and generation: Qdrant identifies the relevant legislation, and the retrieved passages become the evidence supplied to the language model.

In [14]:
print(
    retrieval["context"][:6000]
)


[Fonte 1] EBF — Artigo 44.º-B — Outros benefícios com caráter ambiental atribuídos a imóveis — páginas 76-78
e sejam cumpridos os requisitos de eficiência energética e de qualidade térmica aplicáveis aos edifícios a que se refere o artigo 30.º do Decreto-Lei n.º 118/2013, de 20 de agosto, alterado pelo Decreto- Lei n.º 194/2015, de 14 de setembro, sem prejuízo do disposto no artigo 6.º do Decreto-Lei n.º 53/2014, de 8 de abril. Aditada pela Lei n.º 114/2017, de 29 de dezembro) 2 - Aos imóveis que preencham os requisitos a que se refere o número anterior são aplicáveis os seguintes benefícios fiscais: (Redação da Lei n.º 114/2017, de 29 de dezembro) a) Isenção do imposto municipal sobre imóveis por um período de três anos a contar do ano, inclusive, da conclusão das obras de reabilitação, podendo ser renovado, a requerimento do proprietário, por mais cinco anos no caso de imóveis afetos a arrendamento para habitação permanente ou a habitação própria e permanente; b) Isenção do imposto m

## Teste com várias perguntas

Esta célula permite observar rapidamente se os resultados recuperados fazem sentido semanticamente.


### 5.1 Top-K Retrieval

The number of returned results is controlled through the Top-K parameter.

Retrieving multiple candidates is important in a RAG system because the information required to answer a question may be distributed across several articles or chunks rather than contained in the single highest-ranked result.

In [15]:
test_questions = [
    "Como são tributados os rendimentos prediais?",
    "Quais são as regras relativas ao IVA?",
    "Que benefícios fiscais existem?",
    "Quais são as obrigações declarativas dos sujeitos passivos?",
    "Como é determinado o valor patrimonial tributário?"
]

test_rows = []

for question in test_questions:

    search_results = vector_search(
        question,
        top_k=3
    )

    for rank, result in enumerate(
        search_results,
        start=1
    ):

        payload = result["payload"]

        test_rows.append({
            "question": question,
            "rank": rank,
            "score": result["score"],
            "document_id": payload[
                "document_id"
            ],
            "article": payload[
                "article"
            ],
            "article_title": payload.get(
                "article_title"
            )
        })

df_tests = pd.DataFrame(
    test_rows
)

df_tests


,question,rank,score,document_id,article,article_title
0,Como são tributados os rendimentos prediais?,1,0.704147,CIRS,Artigo 8.º,Rendimentos da categoria F
1,Como são tributados os rendimentos prediais?,2,0.656335,EBF,Artigo 45.º-C,"(Aditado pelo Decreto-Lei n.º 97/2026, de 20 d..."
2,Como são tributados os rendimentos prediais?,3,0.651471,CIRS,Artigo 16.º,As modificações do Código
3,Quais são as regras relativas ao IVA?,1,0.660586,CIVA,Artigo 28.º,Pagamento do imposto liquidado pela administração
4,Quais são as regras relativas ao IVA?,2,0.655285,CIVA,Artigo 12.º,Renúncia à isenção
5,Quais são as regras relativas ao IVA?,3,0.638943,OE2026,Artigo 75.º,Alteração ao Código do Imposto sobre o Valor A...
6,Que benefícios fiscais existem?,1,0.699986,EBF,Artigo 2.º,Conceito de benefício fiscal e de despesa fisc...
7,Que benefícios fiscais existem?,2,0.645418,EBF,Artigo 5.º,Benefícios fiscais automáticos e dependentes d...
8,Que benefícios fiscais existem?,3,0.641075,CIRC,Artigo 92.º,Resultado da liquidação
9,Quais são as obrigações declarativas dos sujei...,1,0.665988,LGT,Artigo 31.º,Obrigações dos sujeitos passivos


## Verificação dos documentos recuperados

Esta vista ajuda a confirmar se o sistema está a selecionar os diplomas esperados para cada pergunta.


### 6.1 Tabular Result Analysis

Search results are converted into a tabular representation to make ranking and metadata easier to compare.

This view is particularly useful when evaluating whether the highest-ranked chunks correspond to the legal provisions expected for a given question.

In [16]:
pd.set_option(
    "display.max_colwidth",
    120
)

df_tests[
    [
        "question",
        "rank",
        "score",
        "document_id",
        "article",
        "article_title"
    ]
]


,question,rank,score,document_id,article,article_title
0,Como são tributados os rendimentos prediais?,1,0.704147,CIRS,Artigo 8.º,Rendimentos da categoria F
1,Como são tributados os rendimentos prediais?,2,0.656335,EBF,Artigo 45.º-C,"(Aditado pelo Decreto-Lei n.º 97/2026, de 20 de maio)"
2,Como são tributados os rendimentos prediais?,3,0.651471,CIRS,Artigo 16.º,As modificações do Código
3,Quais são as regras relativas ao IVA?,1,0.660586,CIVA,Artigo 28.º,Pagamento do imposto liquidado pela administração
4,Quais são as regras relativas ao IVA?,2,0.655285,CIVA,Artigo 12.º,Renúncia à isenção
5,Quais são as regras relativas ao IVA?,3,0.638943,OE2026,Artigo 75.º,Alteração ao Código do Imposto sobre o Valor Acrescentado
6,Que benefícios fiscais existem?,1,0.699986,EBF,Artigo 2.º,Conceito de benefício fiscal e de despesa fiscal e respectivo controlo
7,Que benefícios fiscais existem?,2,0.645418,EBF,Artigo 5.º,Benefícios fiscais automáticos e dependentes de reconhecimento
8,Que benefícios fiscais existem?,3,0.641075,CIRC,Artigo 92.º,Resultado da liquidação
9,Quais são as obrigações declarativas dos sujeitos passivos?,1,0.665988,LGT,Artigo 31.º,Obrigações dos sujeitos passivos


## Resultado

Neste ponto, o pipeline de recuperação semântica está completo:

`Pergunta → bge-m3 → Qdrant → Top-K chunks → contexto`

O notebook seguinte poderá ser dedicado à avaliação quantitativa da recuperação através de métricas como Hit@K, Recall@K e MRR.


## Next Step

This notebook demonstrates that CLARA can retrieve relevant legal passages using semantic similarity, but individual examples are not sufficient to measure retrieval quality systematically.

The next notebook is:

```text
07_retrieval_evaluation.ipynb
```

It evaluates the retrieval component against a predefined set of questions and expected legal sources using quantitative information-retrieval metrics.

After retrieval has been evaluated, the final notebook combines the retrieval pipeline with **AMALIA-9B** to generate grounded natural-language answers.
